In [ ]:
import json
import openpyxl
from openpyxl.styles import Font, PatternFill, Side, Border, Alignment
from openpyxl.utils import get_column_letter

def generate_inventory_report(json_file, excel_file):

    with open(json_file, 'r', encoding= 'utf-8') as f:
        data = json.load(f)

    warehouse_name = data.get("warehouse_name", "")
    inventory_date = data.get("inventory_date", "")
    items = data.get("items", [])

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Inventory Summary"
    ws.merge_cells("A1:F1")
    ws["A1"] = f"{warehouse_name} - Inventory Report" 
    ws["A1"].font = Font(size = 14, bold= True)
    ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws.merge_cells("A2:F2")
    ws["A2"] = f"Date: {inventory_date}"
    ws["A2"].font = Font(size= 12 , bold= True)
    ws["A2"].alignment = Alignment(horizontal= "center", vertical= "center")

    headers = ["Item ID", "Item Name", "Cost Price", "Quantity", "Total Value", "Status"]
    ws.append([])
    ws.append(headers)

    fill_gray = PatternFill(start_color ="D9D9D9", end_color ="D9D9D9", fill_type= "solid")
    fill_green = PatternFill(start_color ="B4F8C8", end_color ="B4F8C8", fill_type= "solid")
    fill_yellow =PatternFill(start_color ="FFFF99", end_color ="FFFF99", fill_type= "solid")

    for col_idx in range (1,7):
        cell = ws.cell(row = 4, column= col_idx)
        cell.font = Font(bold = True)
        cell.fill = fill_gray
        cell.alignment = Alignment(horizontal= "center", vertical= "center")

    total_inventory_val = 0
    for i in items:
        cost_price = i.get("cost_price", 0)
        quantity = i.get("quantity", 0)
        name = i.get("name", "")
        item_id = i.get("item_id", "")

            
        total_value = round(cost_price * quantity, 2)
        total_inventory_val += total_value

        status = ""
        if quantity <= 10:
            status = "Low Stock"
        else:
            status = "In Stock"


        row_data = [item_id, name, cost_price, quantity, total_value, status]
        ws.append(row_data)
        curr_row = ws.max_row

        total_cell = ws.cell(row= curr_row, column = 5)
        if total_value >= 5000:
            total_cell.fill = fill_green
        else:
            total_cell.fill = fill_yellow

    summary_row = ws.max_row + 1
    ws.merge_cells(
        start_row= summary_row, start_column= 1, end_row = summary_row, end_column = 4)

    cell_total = ws.cell(row= summary_row, column= 1, value="Total Inventory Value")
    cell_total.font = Font(bold = True)
    cell_total.alignment = Alignment(horizontal="right", vertical= "center")

    total_val_cell = ws.cell(
        row = summary_row, column= 5, value= total_inventory_val)
    total_val_cell.font = Font(bold= True)

    thin_border = Border(
        right= Side(style="thin"),
        left = Side (style= 'thin'),
        top = Side(style="thin"),
        bottom= Side(style= 'thin')
        )

    for r in range(4, summary_row +1):
        for c in range (1, 7):
            ws.cell(row =r ,column= c).border =thin_border

    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len +4, 12)

    wb.save(excel_file)

In [ ]:
{
  "department": "Global Support",
  "shift_date": "2026-09-07",
  "agents": [
    {"agent_id": "AG-01", "name": "Dan", "calls_handled": 45, "resolved_calls": 40},
    {"agent_id": "AG-02", "name": "Sarah", "calls_handled": 60, "resolved_calls": 57},
    {"agent_id": "AG-03", "name": "Tom", "calls_handled": 30, "resolved_calls": 21},
    {"agent_id": "AG-04", "name": "Maya", "calls_handled": 50, "resolved_calls": 48}
  ]
}

In [ ]:
import json
import openpyxl
from openpyxl.styles import Side, PatternFill, Border, Font, Alignment
from openpyxl.utils import get_column_letter

def generate_support_report(json_file, excel_file):
    with open(json_file, 'r', encoding= 'utf-8') as f:
        data = json.load(f)

    department = data.get("department", "")
    shift_date = data.get("shift_date", "")
    agents = data.get("agents", [])

    thin_border = Border(
        right = Side(style= 'thin'),
        left = Side (style= 'thin'),
        top = Side(style= 'thin'),
        bottom= Side(style='thin')
    )

    fill_gray = PatternFill(start_color="D9D9D9", end_color= "D9D9D9", fill_type= "solid")
    fill_green = PatternFill(start_color="B4F8C8", end_color="B4F8C8", fill_type= "solid")
    fill_yellow =PatternFill(start_color="FFFF99", end_color="FFFF99", fill_type= "solid")

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Support Summary"

    ws.merge_cells("A1:E1")
    ws["A1"] =f"{department}- Daily Performence"
    ws["A1"].font = Font(size =14 , bold=True)
    ws["A1"].alignment = Alignment(horizontal= "center", vertical= "center")
    ws.merge_cells("A2:E2")
    ws["A2"] = f"Date: {shift_date}"
    ws["A2"].font = Font(size =12 ,bold= True)
    ws["A2"].alignment = Alignment(horizontal="center", vertical= "center")

    headers = ["Agent ID", "Agent Name", "Calls Handled", "Resolved Calls", "Success Rate"]
    ws.append([])
    ws.append (headers)

    for col_idx in range(1, 6):
        cell = ws.cell(row = 4, column = col_idx)
        cell.font = Font(bold= True)
        cell.fill = fill_gray
        cell.alignment = Alignment(horizontal= "center", vertical= "center")

        
    total_handled = 0    
    for a in agents:
        agent_id = a.get("agent_id", "")
        name = a.get("name", "")
        calls = a.get("calls_handled", 0)
        resolved = a.get("resolved_calls", 0)

        rate = round((resolved/calls)*100, 2) if calls > 0 else 0
        total_handled += calls

        row_data = [agent_id, name, calls, resolved, rate]
        ws.append(row_data)
        curr_row =ws.max_row

        rate_cell = ws.cell(row= curr_row, column= 5)
        if rate >= 90:
            rate_cell.fill = fill_green
        else:
            rate_cell.fill = fill_yellow

    summary_row = ws.max_row +1
    ws.merge_cells(start_row= summary_row, start_column= 1, end_row= summary_row, end_column= 2)
    label_cell = ws.cell(row = summary_row , column=1, value= "Total Calls Handled")
    label_cell.font = Font(bold= True)
    label_cell.alignment = Alignment(horizontal="right", vertical= "center")

    _total_cell = ws.cell(row = summary_row, column=5, value= total_handled)
    _total_cell.font = Font(bold= True)

    for r in range(4, summary_row+1):
        for c in range (1, 6):
            ws.cell(row =r , column=c).border = thin_border
    for col in ws.columns:
        max_len = max(len(str(cell.value or ''))for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len +4, 12)

    wb.save(excel_file)



In [ ]:
{
  "company_name": "ElectroTech Retail",
  "report_month": "August 2026",
  "branches": [
    {"branch_id": "BR-01", "city": "Tel Aviv", "target": 100000, "actual_sales": 112000},
    {"branch_id": "BR-02", "city": "Haifa", "target": 80000, "actual_sales": 74000},
    {"branch_id": "BR-03", "city": "Jerusalem", "target": 95000, "actual_sales": 98000},
    {"branch_id": "BR-04", "city": "Beer Sheva", "target": 60000, "actual_sales": 55000}
  ]
}


import json
import openpyxl
from openpyxl.styles import Font, PatternFill, Side, Border, Alignment
from openpyxl.utils import get_column_letter

def generate_report(json_file, excel_file):
    # --------------------------------------------------
    # שלב 1: פתיחת קובץ ה-JSON ושליפת שדות ראשיים
    # --------------------------------------------------
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    company_name = data.get("company_name", "")      # כותרת ראשית (שם מחלקה / מחסן)
    report_month = data.get("report_month", "")       # כותרת משנית (תאריך)
    branches = data.get("branches", [])             # רשימת הנתונים (items / agents / projects)

    # --------------------------------------------------
    # שלב 2: יצירת חוברת עבודה ועיצובי בסיס
    # --------------------------------------------------
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Monthly Sales"

    # צבעים
    fill_gray = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
    fill_green = PatternFill(start_color="B4F8C8", end_color="B4F8C8", fill_type="solid")
    fill_yellow = PatternFill(start_color="FFFF99", end_color="FFFF99", fill_type="solid")

    # גבולות דקים
    thin_border = Border(
        top=Side(style="thin"),
        bottom=Side(style="thin"),
        left=Side(style="thin"),
        right=Side(style="thin")
    )

    # --------------------------------------------------
    # שלב 3: כותרות ראשיות (שורות 1, 2) ושורה ריקה (3)
    # הערה: לשנות A1:E1 לפי מספר העמודות בטבלה (E=5, F=6)
    # --------------------------------------------------
    ws.merge_cells("A1:E1")
    ws["A1"] = f"{company_name} - Sales Performance"
    ws["A1"].font = Font(size=14, bold=True)
    ws["A1"].alignment = Alignment(horizontal="center", vertical="center")

    ws.merge_cells("A2:E2")
    ws["A2"] = f"month: {report_month}"
    ws["A2"].font = Font(size=12, bold=True)
    ws["A2"].alignment = Alignment(horizontal="center", vertical="center")

    ws.append([])  # שורה 3 ריקה

    # --------------------------------------------------
    # שלב 4: כותרות הטבלה (שורה 4)
    # --------------------------------------------------
    headers = ["Branch ID", "City", "Target", "Actual Sales", "Met Target"]
    ws.append(headers)

    # עיצוב שורת הכותרת (צבע אפור + הדגשה)
    num_columns = len(headers)
    for col_idx in range(1, num_columns + 1):
        cell = ws.cell(row=4, column=col_idx)
        cell.font = Font(bold=True)
        cell.fill = fill_gray
        cell.alignment = Alignment(horizontal="center", vertical="center")

    # --------------------------------------------------
    # שלב 5 + 6: לולאת הנתונים
    # --------------------------------------------------
    total_sales = 0  # דלי ריק לצבירת הסכום לפני שמתחילים!

    for b in branches:
        # 1. שליפת שדות
        id = b.get("branch_id", "")
        city = b.get("city", "")
        tar = b.get("target", 0)
        sales = b.get("actual_sales", 0)

        if sales >= tar:
            met_target = "Yes"
        else:
            met_target = "No"

        # 3. צבירת סכום עבור שורת הסיכום

        total_sales += sales 

        row_data = [id, city, tar, sales, met_target]
        ws.append(row_data) 

        # 5. צביעת התא של אותה שורה
        curr_row = ws.max_row
        paint_cell = ws.cell(row=curr_row, column=5)  # באיזו עמודה התא שרוצים לצבוע

        if met_target == "Yes":
            paint_cell.fill = fill_green
        else:
            paint_cell.fill = fill_yellow
    # --------------------------------------------------
    # שלב 7: שורת סיכום בסוף הטבלה

    # --------------------------------------------------
    summary_row = ws.max_row + 1

    # מיזוג הטקסט: מ-1 ועד העמודה שלפני המספר (לדוגמה עמודות 1 עד 2)
    ws.merge_cells(start_row=summary_row, start_column=1, end_row=summary_row, end_column=3)

    # כתיבת הטקסט בעמודה 1
    label_cell = ws.cell(row=summary_row, column=1, value="Total / Summary")
    label_cell.font = Font(bold=True)
    label_cell.alignment = Alignment(horizontal="right", vertical="center")

    # כתיבת המספר בעמודה שלו (לדוגמה עמודה 3)
    sum_cell = ws.cell(row=summary_row, column=4, value=total_sales)
    sum_cell.font = Font(bold=True)


    # --------------------------------------------------
    # שלב 8: גבולות דקים (משורה 4 ועד שורת הסיכום)
    # --------------------------------------------------
    for r in range(4, summary_row + 1):
        for c in range(1, num_columns + 1):
            ws.cell(row=r, column=c).border = thin_border

    # --------------------------------------------------
    # שלב 9: רוחב עמודות אוטומטי ושמירה
    # --------------------------------------------------
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

    wb.save(excel_file)

: 